In [2]:
import os
import time
import pandas as pd
import pyperclip
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    ElementClickInterceptedException,
    StaleElementReferenceException
)
from selenium.webdriver.chrome.webdriver import WebDriver

# ... (설정 부분은 이전과 동일) ...
# ==============================================================================
# ✅ 설정 (Configuration)
# ==============================================================================
CONSOLE_URL = 'https://console.thebackend.io/ko/login'
DATA_PAGE_URL = 'https://console.thebackend.io/ko/project/1ea3f14d34e89530ea88b3245bc82dc17d5f52ce1554049f19fce9219a847cfce18bb8891c9ffe90bc65e2b9a3b981853fc5513c1dd200afc9590ba6bfd5fced4230647d25328849e0917641/baseGameInfo/data'
LOGIN_ID = "dksrufp0607@naver.com"
LOGIN_PW = "thdsodydtjdrufp!"

CSV_RANKING_PATH = "C:/엑셀데이터/Ranking_1 (2).csv"
EXCEL_COPY_BASE_PATH = "C:/엑셀데이터/복사"

TARGET_RANKS = [1]
TARGET_USER_IDS = ["j1"]
DATA_TYPES = ["TREASURE_DATA", "GROWTH3_DATA", "BASE_DATA"]

WINDOW_WIDTH = 1100
WINDOW_HEIGHT = 1000

# ==============================================================================
# ✅ 웹 드라이버 및 상호작용 함수 (WebDriver & Interaction Functions)
# ==============================================================================

def setup_browser(position_x: int = 0) -> WebDriver:
    """새로운 크롬 브라우저를 설정하고 반환합니다."""
    driver = webdriver.Chrome()
    driver.set_window_size(WINDOW_WIDTH, WINDOW_HEIGHT)
    driver.set_window_position(position_x, 0)
    return driver

def login_to_console(driver: WebDriver):
    """뒤끝 콘솔에 로그인합니다."""
    try:
        driver.get(CONSOLE_URL)
        id_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "username")))
        pyperclip.copy(LOGIN_ID)
        id_input.send_keys(Keys.CONTROL, 'v')

        pw_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "password")))
        pyperclip.copy(LOGIN_PW)
        pw_input.send_keys(Keys.CONTROL, 'v')
        
        login_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[type="submit"]')))
        login_button.click()
        print("✅ 로그인 완료!")
        WebDriverWait(driver, 15).until(EC.url_changes(CONSOLE_URL)) # 페이지 이동 확인
        time.sleep(2) # ❗️[추가] 로그인 후 다음 페이지가 완전히 로드될 때까지 대기
    except TimeoutException:
        print("⚠️ 로그인 과정에서 시간 초과가 발생했습니다.")
    except Exception as e:
        print(f"⚠️ 로그인 실패: {e}")

def navigate_to_page(driver: WebDriver, url: str):
    """지정된 URL로 이동합니다."""
    driver.get(url)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    print(f"✅ 페이지 로드 완료: {url}")
    time.sleep(1) # ❗️[추가] 페이지 로드 후 스크립트 안정화를 위한 대기

def click_button_by_text(driver: WebDriver, text: str, wait_time: int = 3) -> bool:
    """주어진 텍스트를 포함하는 버튼을 찾아 클릭합니다."""
    print(f"🔍 '{text}' 버튼 탐색 중...")
    try:
        button = WebDriverWait(driver, wait_time).until(
            EC.element_to_be_clickable((By.XPATH, f'//button[contains(text(), "{text}")]'))
        )
        button.click()
        print(f"✅ '{text}' 버튼 클릭 완료!")
        return True
    except TimeoutException:
        print(f"ℹ️ '{text}' 버튼을 찾지 못했습니다. (시간 초과)")
        return False
    except Exception as e:
        print(f"⚠️ '{text}' 버튼 클릭 실패: {e}")
        return False

def execute_search(driver: WebDriver, data_type: str, user_id: str):
    """상세 검색을 통해 특정 유저의 데이터를 찾습니다."""
    try:
        print(f"🔍 '{data_type}' 테이블에서 '{user_id}' 검색 시작...")
        
        table_dropdown = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//div[contains(@class, "_manageInfo-data_1rntr_71")]')))
        table_dropdown.click()
        
        data_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, f'//div[@role="option"][span[text()="{data_type}"]]')))
        data_option.click()
        print(f"✅ '{data_type}' 테이블 선택 완료!")
        time.sleep(2) # ❗️[수정] 테이블 선택 후 관련 UI가 로드될 시간을 충분히 줌

        detail_search_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//p[contains(text(), "상세 검색")]')))
        try:
            detail_search_button.click()
        except ElementClickInterceptedException:
            driver.execute_script("arguments[0].click();", detail_search_button)
        print("✅ 상세 검색 클릭 완료!")
        time.sleep(1) # ❗️[추가] 상세 검색 창이 열릴 때까지 대기

        dropdown = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.ui.selection.dropdown")))
        dropdown.click()
        target_span = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//span[@class="text" and text()="유저 아이디"]')))
        target_span.click()

        search_input = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.NAME, "defaultSearchValue")))
        search_input.clear()
        search_input.send_keys(user_id)
        search_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//button[contains(@class, "ui medium positive button") and @type="submit"]')))
        search_button.click()
        print(f"✅ '{user_id}' 검색 완료!")
        
        # ❗️[수정] 검색 결과가 표시될 시간을 기다린 후 UUID 클릭
        uuid_element = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="gamer_id"]/p')))
        time.sleep(1)
        uuid_element.click()
        print("✅ UUID 클릭 완료!")

    except Exception as e:
        print(f"⚠️ 검색 작업 중 오류 발생: {e}")

def set_ace_editor_value(driver: WebDriver, ace_element, new_value: str):
    """Ace Editor에 값을 설정합니다."""
    try:
        WebDriverWait(driver, 10).until(
            lambda d: d.execute_script("return typeof ace !== 'undefined' && ace.edit(arguments[0]) !== null", ace_element)
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", ace_element)
        driver.execute_script("""
            var editor = ace.edit(arguments[0]);
            editor.setValue(arguments[1], -1);
            editor.session.getUndoManager().reset();
        """, ace_element, str(new_value))
    except Exception as e:
        print(f"⚠️ Ace Editor 값 설정 실패: {e}")

def paste_data(driver: WebDriver, excel_data: list, target_user_id: str):
    """엑셀 데이터를 읽어 웹페이지의 해당 필드에 붙여넣습니다."""
    try:
        edit_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//button[contains(@class, "ui mini primary button") and text()="수정"]')))
        edit_button.click()
        print("✅ 수정 버튼 클릭 완료!")
        time.sleep(2) # ❗️[수정] 수정 모달창이 완전히 로드될 때까지 대기
        
        driver.execute_script("document.body.style.zoom='25%'")
        print("✅ 웹 화면 25% 축소 완료!")
        
        elements = WebDriverWait(driver, 15).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "_columnBorder-edit-modal_1rntr_47")))
        print(f"✅ 수정 모달에서 {len(elements)}개의 필드를 찾았습니다.")
        
        column_element_map = {}
        for element in elements:
            try:
                col_input = element.find_element(By.CSS_SELECTOR, 'input[type="text"][name$=".columnName"]')
                col_name = col_input.get_attribute("value")
                column_element_map[col_name] = element
            except NoSuchElementException:
                continue
        
        processed_count = 0
        for col_name, _, new_value in excel_data:
            if col_name in column_element_map:
                element = column_element_map[col_name]
                try:
                    ace_editor = element.find_element(By.CSS_SELECTOR, "div.ace_editor")
                    set_ace_editor_value(driver, ace_editor, new_value)
                except NoSuchElementException:
                    try:
                        value_field = element.find_element(By.CSS_SELECTOR, 'textarea[name$=".dataValue"]')
                        value_field.clear()
                        value_field.send_keys(str(new_value))
                    except NoSuchElementException:
                        print(f"⚠️ '{col_name}'에 값을 입력할 필드를 찾지 못했습니다.")
                        continue
                processed_count += 1
                # print(f"({processed_count}/{len(excel_data)}) ✅ '{col_name}' 값 입력 완료.") # 너무 많은 로그 출력을 방지하기 위해 주석 처리
            else:
                print(f"⚠️ 웹페이지에서 '{col_name}' 컬럼을 찾을 수 없습니다.")
        
        print(f"✅ 총 {processed_count}개의 데이터 입력 완료.")
        click_button_by_text(driver, "확인")
        time.sleep(2) # ❗️[추가] 데이터 저장 후 페이지가 안정화될 때까지 대기
        print(f"🚀 '{target_user_id}' 유저의 데이터 업데이트 완료!")

    except TimeoutException:
        print("⚠️ '수정' 버튼 또는 데이터 필드를 찾는 데 실패했습니다.")
    except Exception as e:
        print(f"⚠️ 데이터 붙여넣기 중 심각한 오류 발생: {e}")
    finally:
        driver.execute_script("document.body.style.zoom='100%'")
        driver.switch_to.default_content()

# ... (데이터 로드 함수 및 main 함수는 이전과 동일) ...
# ==============================================================================
# ✅ 데이터 로드 함수 (Data Loading Functions)
# ==============================================================================
def load_ranking_info(csv_path: str, rank: int) -> tuple[str, str] | None:
    try:
        df = pd.read_csv(csv_path)
        if 0 <= rank - 1 < len(df):
            row = df.iloc[rank - 1]
            name = row.iloc[2]
            stage = row.iloc[3]
            print(f"ℹ️ {rank}위 정보 로드: Name='{name}', Stage='{stage}'")
            return name, stage
        else:
            print(f"⚠️ CSV 파일에 {rank}위에 해당하는 데이터가 없습니다.")
            return None
    except FileNotFoundError:
        print(f"⚠️ CSV 파일을 찾을 수 없습니다: {csv_path}")
        return None
    except Exception as e:
        print(f"⚠️ CSV 로드 중 오류 발생: {e}")
        return None

def load_excel_data(filename: str) -> list:
    if not os.path.exists(filename):
        print(f"⚠️ 엑셀 파일이 존재하지 않습니다: {filename}")
        return []
    try:
        df = pd.read_excel(filename)
        data = df[["Column Name", "Data Type", "Value"]].values.tolist()
        print(f"✅ '{os.path.basename(filename)}'에서 {len(data)}개의 데이터 로드 완료!")
        return data
    except Exception as e:
        print(f"⚠️ 엑셀 파일 로드 중 오류 발생: {e}")
        return []

# ==============================================================================
# ✅ 메인 실행 (Main Execution)
# ==============================================================================
def main():
    if len(TARGET_RANKS) != len(TARGET_USER_IDS):
        print("⚠️ 설정 오류: TARGET_RANKS와 TARGET_USER_IDS의 개수가 일치해야 합니다.")
        return

    for rank, user_id in zip(TARGET_RANKS, TARGET_USER_IDS):
        print("\n" + "="*50)
        print(f"🚀 작업 시작: {rank}위 데이터 -> 유저 '{user_id}'")
        print("="*50)

        ranking_info = load_ranking_info(CSV_RANKING_PATH, rank)
        if not ranking_info:
            print(f"⏭️ {rank}위 정보 로드 실패. 다음 작업으로 넘어갑니다.")
            continue
        
        source_name, _ = ranking_info
        
        driver = setup_browser()
        try:
            login_to_console(driver)
            
            for data_type in DATA_TYPES:
                print(f"\n--- [ {data_type} ] 데이터 처리 시작 ---")
                
                excel_filename = os.path.join(EXCEL_COPY_BASE_PATH, data_type, f"{source_name}.xlsx")
                excel_data = load_excel_data(excel_filename)
                if not excel_data:
                    print(f"⏭️ 데이터가 없어 '{data_type}' 작업을 건너뜁니다.")
                    continue

                navigate_to_page(driver, DATA_PAGE_URL)
                click_button_by_text(driver, "다음")

                execute_search(driver, data_type, user_id)
                
                paste_data(driver, excel_data, user_id)
        
        except KeyboardInterrupt:
            print("⚠️ 사용자에 의해 프로그램이 중단되었습니다.")
            break
        except Exception as e:
            print(f"🚨 메인 프로세스에서 예기치 않은 오류 발생: {e}")
        finally:
            print("✅ 브라우저를 종료합니다.")
            driver.quit()

if __name__ == "__main__":
    main()


🚀 작업 시작: 1위 데이터 -> 유저 'j1'
ℹ️ 1위 정보 로드: Name='지냉', Stage='1192'
✅ 로그인 완료!

--- [ TREASURE_DATA ] 데이터 처리 시작 ---
✅ '지냉.xlsx'에서 3개의 데이터 로드 완료!
✅ 페이지 로드 완료: https://console.thebackend.io/ko/project/1ea3f14d34e89530ea88b3245bc82dc17d5f52ce1554049f19fce9219a847cfce18bb8891c9ffe90bc65e2b9a3b981853fc5513c1dd200afc9590ba6bfd5fced4230647d25328849e0917641/baseGameInfo/data
🔍 '다음' 버튼 탐색 중...
ℹ️ '다음' 버튼을 찾지 못했습니다. (시간 초과)
🔍 'TREASURE_DATA' 테이블에서 'j1' 검색 시작...
⚠️ 검색 작업 중 오류 발생: Message: element click intercepted: Element <div role="listbox" aria-expanded="false" class="ui dropdown selection icon _manageInfo-data_1rntr_71" tabindex="0">...</div> is not clickable at point (628, 231). Other element would receive the click: <img src="https://s3.ap-northeast-2.amazonaws.com/console.popup.thebackend.io/live/%EB%92%A4%EB%81%9D%20%EC%9A%94%EA%B8%88%EC%A0%9C%20%EA%B8%B0%EC%A4%80%20%ED%99%94%ED%8F%90%20%EC%A0%84%ED%99%98%20%EC%95%88%EB%82%B4%20%282025%EB%85%84%2010%EC%9B%94%20%EC%A0%81%EC%9A%A9%29.png" width